# Notebook 10 — Benchmark: M0 vs M2 on the Public BD Prescription Dataset

In [1]:
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
import random, time
from dataclasses import dataclass
from functools import partial
from pathlib import Path
from collections import defaultdict
import numpy as np, pandas as pd
from PIL import Image
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = (torch.device("mps") if torch.backends.mps.is_available()
          else torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu"))
print(f"Using device: {DEVICE} | torch {torch.__version__}")

Using device: mps | torch 2.12.0


In [2]:
@dataclass
class Config:
    # BD dataset layout (matches your original baseline run)
    data_root: Path = Path("../data/doctors_prescription_bd")
    train_csv: str = "Training/training_labels.csv"; train_img_dir: str = "Training/training_words"
    val_csv: str = "Validation/validation_labels.csv"; val_img_dir: str = "Validation/validation_words"
    test_csv: str = "Testing/testing_labels.csv"; test_img_dir: str = "Testing/testing_words"
    img_col: str = "IMAGE"; label_col: str = "MEDICINE_NAME"

    img_height: int = 48; img_width: int = 320          # same preprocessing as custom run
    rnn_hidden: int = 256; rnn_layers: int = 2; dropout: float = 0.2
    batch_size: int = 64; epochs: int = 80; lr: float = 1e-4
    warmup_epochs: int = 5; weight_decay: float = 1e-4; grad_clip: float = 5.0
    early_stop_patience: int = 15; num_workers: int = 0
    best_tau: float = 0.5                                # lexicon threshold (sweep below)
    ckpt_dir: Path = Path("../checkpoints/benchmark_bd")

CFG = Config()
CFG.ckpt_dir.mkdir(parents=True, exist_ok=True)

## 1. Shared components (same Vocab/Dataset/CRNN/metrics as the custom pipeline)

In [3]:
class Vocab:
    BLANK = 0
    def __init__(self, texts):
        chars = sorted(set("".join(texts)))
        self.idx2char = {i + 1: c for i, c in enumerate(chars)}
        self.char2idx = {c: i for i, c in self.idx2char.items()}
    def __len__(self): return len(self.idx2char) + 1
    def encode(self, t): return [self.char2idx[c] for c in t]
    def decode_greedy(self, indices):
        out, prev = [], None
        for i in indices:
            if i != prev and i != self.BLANK:
                out.append(self.idx2char[i])
            prev = i
        return "".join(out)

class WordImageDataset(Dataset):
    def __init__(self, csv_path, img_dir, cfg, vocab=None, augment=False):
        self.df = pd.read_csv(csv_path).dropna(subset=[cfg.label_col])
        self.df[cfg.label_col] = self.df[cfg.label_col].astype(str).str.strip().str.lower()
        self.img_dir = Path(img_dir); self.cfg = cfg; self.vocab = vocab; self.augment = augment
    def labels(self): return self.df[self.cfg.label_col].tolist()
    def __len__(self): return len(self.df)
    def _load(self, path):
        img = Image.open(path).convert("L"); w, h = img.size
        new_w = min(max(1, int(round(w * self.cfg.img_height / h))), self.cfg.img_width)
        img = img.resize((new_w, self.cfg.img_height), Image.BILINEAR)
        canvas = Image.new("L", (self.cfg.img_width, self.cfg.img_height), color=255)
        canvas.paste(img, (0, 0)); return canvas
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = self._load(self.img_dir / str(row[self.cfg.img_col]))
        if self.augment:
            img = img.rotate(random.uniform(-3, 3), resample=Image.BILINEAR, fillcolor=255)
        x = torch.from_numpy(np.array(img, dtype=np.float32) / 255.0).unsqueeze(0)
        target = torch.tensor(self.vocab.encode(row[self.cfg.label_col]), dtype=torch.long)
        return x, target, row[self.cfg.label_col], str(row[self.cfg.img_col])

def collate(batch):
    xs, targets, texts, fnames = zip(*batch)
    tl = torch.tensor([t.numel() for t in targets], dtype=torch.long)
    return torch.stack(xs), torch.cat(targets), tl, list(texts), list(fnames)

def edit_distance(a, b):
    if a == b: return 0
    if not a: return len(b)
    if not b: return len(a)
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        curr = [i]
        for j, cb in enumerate(b, 1):
            curr.append(min(prev[j] + 1, curr[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = curr
    return prev[-1]

def corpus_metrics(preds, refs):
    tot = sum(edit_distance(p, r) for p, r in zip(preds, refs))
    chars = sum(len(r) for r in refs); exact = sum(p == r for p, r in zip(preds, refs))
    return {"CER": tot / max(chars, 1), "ExactMatch": exact / len(refs), "n": len(refs)}

class CRNN(nn.Module):
    def __init__(self, n_classes, rnn_hidden=256, rnn_layers=2, dropout=0.2):
        super().__init__()
        def conv(i, o, bn=False):
            L = [nn.Conv2d(i, o, 3, 1, 1)]
            if bn: L.append(nn.BatchNorm2d(o))
            L.append(nn.ReLU(inplace=True)); return L
        self.cnn = nn.Sequential(
            *conv(1, 64), nn.MaxPool2d(2, 2), *conv(64, 128), nn.MaxPool2d(2, 2),
            *conv(128, 256), *conv(256, 256), nn.MaxPool2d((2, 1), (2, 1)),
            *conv(256, 512, bn=True), *conv(512, 512, bn=True), nn.MaxPool2d((2, 1), (2, 1)),
        )
        self.collapse = nn.AdaptiveAvgPool2d((1, None))
        self.rnn = nn.LSTM(512, rnn_hidden, rnn_layers, bidirectional=True,
                           dropout=dropout if rnn_layers > 1 else 0.0)
        self.head = nn.Linear(2 * rnn_hidden, n_classes)
    def forward(self, x):
        f = self.collapse(self.cnn(x)).squeeze(2).permute(2, 0, 1)
        seq, _ = self.rnn(f)
        return self.head(seq)

## 2. Data

In [4]:
train_ds = WordImageDataset(CFG.data_root / CFG.train_csv, CFG.data_root / CFG.train_img_dir, CFG, augment=True)
VOCAB = Vocab(train_ds.labels()); train_ds.vocab = VOCAB
val_ds = WordImageDataset(CFG.data_root / CFG.val_csv, CFG.data_root / CFG.val_img_dir, CFG, vocab=VOCAB)
test_ds = WordImageDataset(CFG.data_root / CFG.test_csv, CFG.data_root / CFG.test_img_dir, CFG, vocab=VOCAB)
print(f"train={len(train_ds)} val={len(val_ds)} test={len(test_ds)} vocab={len(VOCAB)}")

_collate = collate
train_dl = DataLoader(train_ds, CFG.batch_size, shuffle=True, num_workers=CFG.num_workers,
                      collate_fn=_collate, drop_last=True)
val_dl = DataLoader(val_ds, CFG.batch_size, shuffle=False, num_workers=CFG.num_workers, collate_fn=_collate)
test_dl = DataLoader(test_ds, CFG.batch_size, shuffle=False, num_workers=CFG.num_workers, collate_fn=_collate)

train=3120 val=780 test=780 vocab=26


## 3. Train the baseline CRNN on BD

In [5]:
def greedy(model, loader):
    model.eval(); preds, refs, files = [], [], []
    with torch.no_grad():
        for xb, _, _, texts, fnames in loader:
            logits = model(xb.to(DEVICE))
            idx = logits.argmax(-1).permute(1, 0).cpu()
            preds += [VOCAB.decode_greedy(s.tolist()) for s in idx]
            refs += texts; files += fnames
    return preds, refs, files

def train(model):
    model.to(DEVICE)
    ctc = nn.CTCLoss(blank=Vocab.BLANK, zero_infinity=True)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=4)
    best, no_imp = float("inf"), 0
    for epoch in range(1, CFG.epochs + 1):
        if epoch <= CFG.warmup_epochs:
            for g in opt.param_groups: g["lr"] = CFG.lr * epoch / CFG.warmup_epochs
        model.train(); t0 = time.time()
        for xb, targets, tlens, _, _ in train_dl:
            xb = xb.to(DEVICE); logits = model(xb); T = logits.shape[0]
            loss = ctc(logits.log_softmax(2).cpu(), targets,
                       torch.full((xb.shape[0],), T, dtype=torch.long), tlens)
            opt.zero_grad(set_to_none=True); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), CFG.grad_clip); opt.step()
        vp, vr, _ = greedy(model, val_dl); vcer = corpus_metrics(vp, vr)["CER"]
        if epoch > CFG.warmup_epochs: sched.step(vcer)
        if epoch % 5 == 0 or epoch == 1:
            print(f"epoch {epoch:3d} | val CER {vcer:.4f} | {time.time()-t0:.1f}s")
        if vcer < best:
            best, no_imp = vcer, 0
            torch.save({"model": model.state_dict()}, CFG.ckpt_dir / "best.pt")
        else:
            no_imp += 1
            if no_imp >= CFG.early_stop_patience:
                print(f"early stop at epoch {epoch} (best val CER {best:.4f})"); break

model = CRNN(len(VOCAB), CFG.rnn_hidden, CFG.rnn_layers, CFG.dropout)
print(f"params: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")
train(model)

params: 7.67M
epoch   1 | val CER 1.0000 | 19.5s
epoch   5 | val CER 0.8198 | 15.4s
epoch  10 | val CER 0.7704 | 15.2s
epoch  15 | val CER 0.6646 | 15.2s
epoch  20 | val CER 0.5951 | 15.2s
epoch  25 | val CER 0.5601 | 16.2s
epoch  30 | val CER 0.4132 | 15.4s
epoch  35 | val CER 0.3567 | 15.5s
epoch  40 | val CER 0.2927 | 15.5s
epoch  45 | val CER 0.1840 | 15.5s
epoch  50 | val CER 0.1516 | 15.5s
epoch  55 | val CER 0.1111 | 15.5s
epoch  60 | val CER 0.0978 | 15.8s
epoch  65 | val CER 0.0868 | 15.6s
epoch  70 | val CER 0.0834 | 15.6s
epoch  75 | val CER 0.0842 | 15.6s
early stop at epoch 79 (best val CER 0.0820)


## 4. Build BD lexicon (training labels only) + lexicon decode

In [6]:
bd_lexicon = sorted(set(pd.read_csv(CFG.data_root / CFG.train_csv)[CFG.label_col]
                        .astype(str).str.strip().str.lower()))
by_len = defaultdict(list)
for w in bd_lexicon: by_len[len(w)].append(w)
print(f"BD lexicon size: {len(bd_lexicon)} (closed vocabulary)")

def nearest(word, k=1, gap=3):
    if not word: return None, 10**9
    if word in by_len.get(len(word), ()): return word, 0
    best, bd = None, 10**9
    for L in range(len(word)-gap, len(word)+gap+1):
        for e in by_len.get(L, ()):
            d = edit_distance(word, e)
            if d < bd: best, bd = e, d
            if bd == 1: return best, bd
    return best, bd

def lexicon_decode(raw, tau):
    out = []
    for p in raw:
        e, d = nearest(p)
        out.append(e if (e is not None and d/max(len(p),1) <= tau) else p)
    return out

BD lexicon size: 78 (closed vocabulary)


## 5. Evaluate M0 vs M2 on BD test (τ swept on validation)

In [7]:
ck = torch.load(CFG.ckpt_dir / "best.pt", map_location="cpu")
model.load_state_dict(ck["model"]); model.to(DEVICE)

val_raw, val_refs, _ = greedy(model, val_dl)
test_raw, test_refs, _ = greedy(model, test_dl)

# sweep tau on validation
best_tau, best_em = 0.0, -1
for tau in [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]:
    em = corpus_metrics(lexicon_decode(val_raw, tau), val_refs)["ExactMatch"]
    if em > best_em: best_em, best_tau = em, tau
print(f"selected τ={best_tau} (val EM {best_em:.4f})")

m0 = corpus_metrics(test_raw, test_refs)
m2 = corpus_metrics(lexicon_decode(test_raw, best_tau), test_refs)
print(f"\nBD BENCHMARK — test (n={m0['n']}):")
print(f"  M0 baseline   : CER {m0['CER']:.4f} | EM {m0['ExactMatch']:.4f}")
print(f"  M2 +lexicon   : CER {m2['CER']:.4f} | EM {m2['ExactMatch']:.4f}")

# safety
fix = broke = 0
dec = lexicon_decode(test_raw, best_tau)
for raw, d, r in zip(test_raw, dec, test_refs):
    if raw != d:
        if raw != r and d == r: fix += 1
        elif raw == r and d != r: broke += 1
print(f"  lexicon: {fix} fixes, {broke} breaks")

pd.DataFrame([{"dataset": "BD", "model": "M0", **m0},
              {"dataset": "BD", "model": "M2", **m2}]).to_csv(CFG.ckpt_dir / "bd_results.csv", index=False)

selected τ=0.5 (val EM 0.9064)

BD BENCHMARK — test (n=780):
  M0 baseline   : CER 0.1498 | EM 0.6359
  M2 +lexicon   : CER 0.1117 | EM 0.8449
  lexicon: 163 fixes, 0 breaks


## 6. For the thesis
- Report BD M0 vs M2 alongside your custom-data M0 vs M2. Two prescription datasets, same
  method — that's the generality argument the brief asks for.
- Expected contrast: on BD (closed-vocabulary, 78 names) the baseline is already strong and
  the lexicon helps less but does no harm (few/zero breaks). On your custom data (open-
  vocabulary, 1500+ names) the lexicon helps a lot. This contrast IS the insight: lexicon
  decoding's value scales with vocabulary openness — exactly when domain knowledge matters.
- Sanity check: your BD CER here should be broadly comparable to your very first BD baseline
  run (CER ~0.15), confirming the pipeline reproduces.